In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.mplot3d.art3d import Poly3DCollection, Line3DCollection
from mpl_toolkits.mplot3d import proj3d
import seaborn as sns

sns.set_theme(style='whitegrid', font_scale=0.95)


class Arrow3D(FancyArrowPatch):
    def __init__(self, xs, ys, zs, *args, **kwargs):
        super().__init__((0, 0), (0, 0), *args, **kwargs)
        self._verts3d = xs, ys, zs

    def do_3d_projection(self, renderer=None):
        xs3d, ys3d, zs3d = self._verts3d
        xs, ys, zs = proj3d.proj_transform(xs3d, ys3d, zs3d, self.axes.M)
        self.set_positions((xs[0], ys[0]), (xs[1], ys[1]))
        return min(zs)


# ── palette ───────────────────────────────────────────────────────────────────
CRIMSON = '#73102A'   # P point
INDIGO  = '#1C0B9C'   # Q point
OLIVE   = '#558C03'   # simplex face
TEAL    = '#038C7F'   # vertices
DIST_E  = '#D9A404'   # Euclidean distance (amber/gold)
DIST_A  = '#9C27B0'   # Aitchison distance (purple)

# Axis colors
A_COLOR = '#CC4400'   # Axis A (deep orange-red)
B_COLOR = '#007700'   # Axis B (forest green)
C_COLOR = '#0044BB'   # Axis C (medium blue)

# ── helper functions ─────────────────────────────────────────────────────────
def _clr(v):
    v = np.asarray(v, float)
    return np.log(v) - np.mean(np.log(v))


def draw_point_box(ax, x, y, z, color, lw=0.7, edge_alpha=0.45, face_alpha=0.07):
    """Draw a transparent bounding box from the origin to point (x, y, z)."""
    # Wireframe edges (excluding the 3 already covered by axis arrows)
    edges = [
        # floor (z=0)
        [(x,0,0),(x,y,0)], [(0,y,0),(x,y,0)],
        # ceiling (z=z)
        [(0,0,z),(x,0,z)], [(0,0,z),(0,y,z)],
        [(x,0,z),(x,y,z)], [(0,y,z),(x,y,z)],
        # vertical pillars (not on axes)
        [(x,0,0),(x,0,z)], [(0,y,0),(0,y,z)], [(x,y,0),(x,y,z)],
    ]
    for (x0,y0,z0),(x1,y1,z1) in edges:
        ax.plot([x0,x1],[y0,y1],[z0,z1],
                color=color, lw=lw, alpha=edge_alpha, ls=':', zorder=2)

    # Transparent filled faces
    faces = [
        [(0,0,0),(x,0,0),(x,y,0),(0,y,0)],   # bottom
        [(0,0,z),(x,0,z),(x,y,z),(0,y,z)],   # top
        [(0,y,0),(x,y,0),(x,y,z),(0,y,z)],   # front (far-y wall)
        [(0,0,0),(x,0,0),(x,0,z),(0,0,z)],   # back  (near-y wall)
        [(0,0,0),(0,y,0),(0,y,z),(0,0,z)],   # left  (near-x wall)
        [(x,0,0),(x,y,0),(x,y,z),(x,0,z)],   # right (far-x wall)
    ]
    poly = Poly3DCollection(faces, alpha=face_alpha,
                             facecolor=color, edgecolor='none', zorder=1)
    ax.add_collection3d(poly)


# ── data ──────────────────────────────────────────────────────────────────────
a, b, c = 2, 2, 1     # P
q, r, s = 1, 1, 3     # Q
a_close, b_close, c_close = a/(a+b+c), b/(a+b+c), c/(a+b+c)
q_close, r_close, s_close = q/(q+r+s), r/(q+r+s), s/(q+r+s)

d_E = float(np.linalg.norm([a - q, b - r, c - s]))
d_A = float(np.linalg.norm(_clr([a, b, c]) - _clr([q, r, s])))

# ── figure ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(16, 5.2), dpi=150, facecolor='white')
gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.30,
                        width_ratios=[1, 1.05, 1.05])

# ═════════════════════════════════════════════════════════════════════════════
# PANEL 1 — Point in Euclidean space
# ═════════════════════════════════════════════════════════════════════════════
ax1 = fig.add_subplot(gs[0], projection='3d')

AX_MAX1 = 3.3

# ── colored axis arrows representing A, B, C ──
for xs, ys, zs, col, lbl, tp in [
    ([0, AX_MAX1], [0, 0],       [0, 0],       A_COLOR, 'A', (AX_MAX1+0.12, 0, 0)),
    ([0, 0],       [0, AX_MAX1], [0, 0],       B_COLOR, 'B', (0, AX_MAX1+0.12, 0)),
    ([0, 0],       [0, 0],       [0, AX_MAX1], C_COLOR, 'C', (0, 0, AX_MAX1+0.12)),
]:
    arr = Arrow3D(xs, ys, zs, arrowstyle="-|>", mutation_scale=13, lw=1.8, color=col, zorder=3)
    ax1.add_artist(arr)
    ax1.text(*tp, lbl, fontsize=9, fontweight='bold', color=col)

# ── transparent bounding boxes for P and Q ──
draw_point_box(ax1, a, b, c, CRIMSON)
draw_point_box(ax1, q, r, s, INDIGO)

# ── arrows from origin to P and Q ──
for xs, ys, zs, col in [
    ([0, a], [0, b], [0, c], CRIMSON),
    ([0, q], [0, r], [0, s], INDIGO),
]:
    arr = Arrow3D(xs, ys, zs, arrowstyle="-|>", mutation_scale=13, lw=1.6,
                  color=col, linestyle='dashed', zorder=5)
    ax1.add_artist(arr)

# ── points ──
ax1.scatter([a],[b],[c], color=CRIMSON, s=80, zorder=10, edgecolors='black', linewidths=0.6)
ax1.text(a + 0.1, b - 0.1, c - 0.5, f'P({a}, {b}, {c})',
         fontsize=7.5, fontweight='bold', color=CRIMSON)

ax1.scatter([q],[r],[s], color=INDIGO, s=80, zorder=10, edgecolors='black', linewidths=0.6)
ax1.text(q + 0.02, r - 0.02, s + 0.4, f'Q({q}, {r}, {s})',
         fontsize=7.5, fontweight='bold', color=INDIGO)

# ── Euclidean distance line ──
ax1.plot([a, q],[b, r],[c, s], color=DIST_E, lw=1.8, zorder=6)
mx, my, mz = (a+q)/2, (b+r)/2, (c+s)/2
ax1.text(mx - 0.04, my + 0.09, mz + 0.02,
         f'$d_E$ = {d_E:.3f}', fontsize=7.5, color=DIST_E)

ax1.set_xlim(0, AX_MAX1+0.5); ax1.set_ylim(0, AX_MAX1+0.5); ax1.set_zlim(0, AX_MAX1+0.5)
ax1.set_xlabel(''); ax1.set_ylabel(''); ax1.set_zlabel('')
ax1.set_title('1. Point in Euclidean space', fontsize=10, fontweight='bold', pad=8)
ax1.view_init(elev=22, azim=-28)

# ═════════════════════════════════════════════════════════════════════════════
# PANEL 2 — Normalise onto the simplex
# ═════════════════════════════════════════════════════════════════════════════
ax2 = fig.add_subplot(gs[1], projection='3d')

V = np.array([[1,0,0],[0,1,0],[0,0,1]], float)
ax2.add_collection3d(Poly3DCollection([V.tolist()], alpha=0.30,
                                       facecolor=OLIVE, edgecolor=OLIVE, linewidths=1.5))

# ── colored axis arrows representing A%, B%, C% ──
AX_MAX2 = 1.25
for xs, ys, zs, col, lbl, tp in [
    ([0, AX_MAX2], [0, 0],       [0, 0],       A_COLOR, 'A%', (AX_MAX2+0.06, 0, 0)),
    ([0, 0],       [0, AX_MAX2], [0, 0],       B_COLOR, 'B%', (0, AX_MAX2+0.06, 0)),
    ([0, 0],       [0, 0],       [0, AX_MAX2], C_COLOR, 'C%', (0, 0, AX_MAX2+0.06)),
]:
    arr = Arrow3D(xs, ys, zs, arrowstyle="-|>", mutation_scale=13, lw=1.8, color=col, zorder=3)
    ax2.add_artist(arr)
    ax2.text(*tp, lbl, fontsize=9, fontweight='bold', color=col)

# ── lines from origin to P and Q (normalized) ──
ax2.plot([0, a_close],[0, b_close],[0, c_close], color=CRIMSON, lw=1.5, ls='--', zorder=5)
ax2.plot([0, q_close],[0, r_close],[0, s_close], color=INDIGO, lw=1.5, ls='--', zorder=5)

# ── distance line between normalized P and Q ──
ax2.plot([a_close,q_close],[b_close,r_close],[c_close,s_close], color=DIST_E, lw=1.6, zorder=6)

# ── simplex vertices ──
for pt, lbl in zip(V, ['A','B','C']):
    ax2.scatter(*pt, color=TEAL, s=70, zorder=10, edgecolors='black', linewidths=0.5)
    ax2.text(pt[0]+0.06, pt[1]+0.06, pt[2]+0.06, lbl,
             fontsize=11, fontweight='bold', color=TEAL)

# ── points ──
ax2.scatter([a_close],[b_close],[c_close], color=CRIMSON, s=80, zorder=11, edgecolors='black', linewidths=0.5)
ax2.text(a_close + 0.06, b_close + 0.05, c_close - 0.05,
         f'P\'({a_close:.1f},{b_close:.1f},{c_close:.1f})',
         fontsize=7, fontweight='bold', color=CRIMSON)

ax2.scatter([q_close],[r_close],[s_close], color=INDIGO, s=80, zorder=11, edgecolors='black', linewidths=0.5)
ax2.text(q_close + 0.05, r_close + 0.05, s_close - 0.04,
         f'Q\'({q_close:.1f},{r_close:.1f},{s_close:.1f})',
         fontsize=7, fontweight='bold', color=INDIGO)

ax2.set_xlabel(''); ax2.set_ylabel(''); ax2.set_zlabel('')
ax2.set_title('2. Normalise onto the simplex\n$a + b + c = 1$',
              fontsize=10, fontweight='bold', pad=8)
ax2.view_init(elev=22, azim=-28)

# ═════════════════════════════════════════════════════════════════════════════
# PANEL 3 — Ternary (2-D) diagram
# ═════════════════════════════════════════════════════════════════════════════
ax3 = fig.add_subplot(gs[2])
ax3.set_aspect('equal')
ax3.set_axis_off()

TA = np.array([0.5, np.sqrt(3)/2])
TB = np.array([0.0, 0.0])
TC = np.array([1.0, 0.0])

def t2d(p_, q_, r_):
    return p_*TA + q_*TB + r_*TC

tri_patch = plt.Polygon([TA, TB, TC], facecolor=OLIVE, edgecolor=OLIVE,
                          lw=1.8, alpha=0.28, zorder=1)
ax3.add_patch(tri_patch)

GRID_C = '#cccccc'
for v in [0.2, 0.4, 0.6, 0.8]:
    p1, p2 = t2d(v, 1-v, 0), t2d(v, 0, 1-v)
    ax3.plot([p1[0],p2[0]], [p1[1],p2[1]], color=GRID_C, lw=0.6, zorder=2)
    p1, p2 = t2d(1-v, v, 0), t2d(0, v, 1-v)
    ax3.plot([p1[0],p2[0]], [p1[1],p2[1]], color=GRID_C, lw=0.6, zorder=2)
    p1, p2 = t2d(1-v, 0, v), t2d(0, 1-v, v)
    ax3.plot([p1[0],p2[0]], [p1[1],p2[1]], color=GRID_C, lw=0.6, zorder=2)

for v in [0.2, 0.4, 0.6, 0.8]:
    pt = t2d(v, 0, 1-v)
    ax3.text(pt[0]+0.025, pt[1], f'{v}', fontsize=6.5, color='gray', ha='left', va='center')
    pt = t2d(0, v, 1-v)
    ax3.text(pt[0], pt[1]-0.045, f'{v}', fontsize=6.5, color='gray', ha='center', va='top')
    pt = t2d(v, 1-v, 0)
    ax3.text(pt[0]-0.025, pt[1], f'{v}', fontsize=6.5, color='gray', ha='right', va='center')

# ── colored axis arrows (0% → 100%) ──
OFFSET = 0.09

# A% — right edge (TC → TA)
right_dir  = (TA - TC) / np.linalg.norm(TA - TC)
right_norm = np.array([right_dir[1], -right_dir[0]])
sA = TC + OFFSET * right_norm
eA = TA + OFFSET * right_norm
ax3.annotate("", xy=eA, xytext=sA,
             arrowprops=dict(arrowstyle="-|>", color=A_COLOR, lw=2.0, mutation_scale=14))
ax3.text((sA[0]+eA[0])/2 + 0.05*right_norm[0],
         (sA[1]+eA[1])/2 + 0.05*right_norm[1],
         'A %', fontsize=9, fontweight='bold', color=A_COLOR,
         ha='left', va='center', rotation=-60)

# B% — bottom edge (TC → TB)
bot_norm = np.array([0.0, -1.2])
sB = TC + OFFSET * bot_norm
eB = TB + OFFSET * bot_norm
ax3.annotate("", xy=eB, xytext=sB,
             arrowprops=dict(arrowstyle="-|>", color=B_COLOR, lw=2.0, mutation_scale=14))
ax3.text((sB[0]+eB[0])/2,
         (sB[1]+eB[1])/2 - 0.05,
         'B %', fontsize=9, fontweight='bold', color=B_COLOR,
         ha='center', va='top')

# C% — left edge (TB → TA)
left_dir  = (TA - TB) / np.linalg.norm(TA - TB)
left_norm = np.array([-left_dir[1], left_dir[0]])
sC = TB + OFFSET * left_norm
eC = TA + OFFSET * left_norm
ax3.annotate("", xy=eC, xytext=sC,
             arrowprops=dict(arrowstyle="-|>", color=C_COLOR, lw=2.0, mutation_scale=14))
ax3.text((sC[0]+eC[0])/2 + 0.05*left_norm[0],
         (sC[1]+eC[1])/2 + 0.05*left_norm[1],
         'C %', fontsize=9, fontweight='bold', color=C_COLOR,
         ha='right', va='center', rotation=60)

# ── vertices ──
for pt, lbl, ha, va, xo, yo in zip(
    [TA, TB, TC], ['A', 'B', 'C'],
    ['center','right','left'], ['bottom','top','top'],
    [0, -0.05, 0.05], [0.06, -0.05, -0.05]):
    ax3.plot(*pt, 'o', color=TEAL, markersize=9,
             markeredgecolor='black', markeredgewidth=0.5, zorder=5)
    ax3.text(pt[0]+xo, pt[1]+yo, lbl, fontsize=12, fontweight='bold',
             color=TEAL, ha=ha, va=va)

# ── lines from TB origin to P2 and Q2 ──
P2 = t2d(a_close, b_close, c_close)
Q2 = t2d(q_close, r_close, s_close)
ax3.plot([TB[0], P2[0]], [TB[1], P2[1]], color=CRIMSON, lw=1.3, ls='--', zorder=4, alpha=0.7)
ax3.plot([TB[0], Q2[0]], [TB[1], Q2[1]], color=INDIGO,  lw=1.3, ls='--', zorder=4, alpha=0.7)

# ── points P and Q ──
ax3.plot(*P2, 'o', color=CRIMSON, markersize=8,
         markeredgecolor='black', markeredgewidth=0.5, zorder=10)
ax3.text(P2[0]-0.06, P2[1]+0.06, f'P\'({a_close},{b_close},{c_close})',
         fontsize=7.5, fontweight='bold', color=CRIMSON)

ax3.plot(*Q2, 'o', color=INDIGO, markersize=8,
         markeredgecolor='black', markeredgewidth=0.5, zorder=10)
ax3.text(Q2[0]+0.04, Q2[1]+0.05, f'Q\'({q_close},{r_close},{s_close})',
         fontsize=7.5, fontweight='bold', color=INDIGO)

# ── Aitchison distance line ──
ax3.plot([P2[0], Q2[0]], [P2[1], Q2[1]], color=DIST_A, lw=1.8, zorder=5)
tm = (P2 + Q2) / 2
ax3.text(tm[0] + 0.04, tm[1] + 0.04, f'$d_A$ = {d_A:.3f}',
         fontsize=7.5, color=DIST_A, ha='center')

ax3.set_xlim(-0.26, 1.32)
ax3.set_ylim(-0.26, 1.05)
ax3.set_title('3. Ternary (2-D) diagram', fontsize=10, fontweight='bold', pad=8)

# ── figure title and legend ──────────────────────────────────────────────────
fig.suptitle('From Euclidean Space to Compositional Space',
             fontsize=13, fontweight='bold', y=1.02)

legend_elements = [
    Line2D([0],[0], color=CRIMSON, marker='o', ls='--', markersize=6,
           label=f'Sample P  ({a}, {b}, {c})'),
    Line2D([0],[0], color=INDIGO,  marker='o', ls='--', markersize=6,
           label=f'Sample Q  ({q}, {r}, {s})'),
    Line2D([0],[0], color=DIST_E,  lw=2, ls='-',
           label=f'Euclidean $d_E$ = {d_E:.3f}'),
    Line2D([0],[0], color=DIST_A,  lw=2, ls='-',
           label=f'Aitchison $d_A$ = {d_A:.3f}'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.14), fontsize=8.5, frameon=True,
           edgecolor='#cccccc')

plt.tight_layout()
plt.savefig('../figures/1_cartesian_to_ternary.png', dpi=300, bbox_inches='tight', pad_inches=0.12)
plt.show()
